In [1]:
import os
import joblib
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.neural_network import MLPClassifier

SEED = 42
SR = 16000
MANIFEST = "/home/feliciano/dataset_manifest.csv"
GROUP_SPLIT = "/home/feliciano/group_split.csv"
BACKGROUND_FOLDER = "/media/feliciano/Aux/AI_AFS_DATASET/AFS_BEHAVIOUR_DATASET/background"

fish_df = pd.read_csv(MANIFEST)
split_df = pd.read_csv(GROUP_SPLIT)
fish_df = fish_df.merge(split_df, on="parent_file_id", how="inner")
fish_df = fish_df[fish_df["label"].str.lower().isin(["normal","clustering","agitation"])].copy()

background_files = [os.path.join(BACKGROUND_FOLDER,f) for f in os.listdir(BACKGROUND_FOLDER) if f.lower().endswith('.wav')]
background_df = pd.DataFrame({"clip_path":background_files,"label":"background"})

bg_train, bg_test = train_test_split(background_df,test_size=0.20,random_state=SEED,shuffle=True)
bg_train['split']='train'
bg_test['split']='test'
background_df = pd.concat([bg_train,bg_test],ignore_index=True)

df = pd.concat([fish_df,background_df],ignore_index=True)

def extract_features(path):
    signal, sr = librosa.load(path, sr=SR, mono=True)
    rms = np.mean(librosa.feature.rms(y=signal))
    signal_z = (signal - np.mean(signal)) / (np.std(signal)+1e-8)
    features=[rms]
    features.append(np.mean(librosa.feature.zero_crossing_rate(signal_z)))
    features.append(np.mean(librosa.feature.spectral_centroid(y=signal_z,sr=sr)))
    features.append(np.mean(librosa.feature.spectral_bandwidth(y=signal_z,sr=sr)))
    features.append(np.mean(librosa.feature.spectral_rolloff(y=signal_z,sr=sr)))
    contrast = librosa.feature.spectral_contrast(y=signal_z,sr=sr,n_bands=6)
    features.extend(np.mean(contrast,axis=1))
    mfcc = librosa.feature.mfcc(y=signal_z,sr=sr,n_mfcc=20)
    features.extend(np.mean(mfcc,axis=1))
    return np.array(features,dtype=np.float32)

X=[]
y=[]
for idx,row in df.iterrows():
    if idx % 1000 == 0:
        print(f"{idx}/{len(df)}")
    X.append(extract_features(row['clip_path']))
    y.append(row['label'])

X=np.vstack(X)
encoder=LabelEncoder()
y=encoder.fit_transform(y)

train_mask=(df['split']=='train')
test_mask=(df['split']=='test')

X_train=X[train_mask]
X_test=X[test_mask]
y_train=y[train_mask]
y_test=y[test_mask]

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

model = MLPClassifier(

    hidden_layer_sizes=(256,128),

    activation='relu',

    solver='adam',

    alpha=0.0001,

    batch_size=256,

    learning_rate_init=0.001,

    max_iter=500,

    random_state=SEED

)

print('Training MLP...')

model.fit(X_train,y_train)

pred=model.predict(X_test)

accuracy=accuracy_score(y_test,pred)
macro_f1=f1_score(y_test,pred,average='macro')
weighted_f1=f1_score(y_test,pred,average='weighted')
per_class_f1=f1_score(y_test,pred,average=None)

print('Accuracy:',accuracy)
print('Macro F1:',macro_f1)
print('Weighted F1:',weighted_f1)
print(classification_report(y_test,pred,target_names=encoder.classes_))

pd.DataFrame({
'clip_path':df.loc[test_mask,'clip_path'].values,
'true_label':encoder.inverse_transform(y_test),
'predicted_label':encoder.inverse_transform(pred)
}).to_csv('mlp_background_predictions.csv',index=False)

cm=confusion_matrix(y_test,pred)
pd.DataFrame(cm,index=encoder.classes_,columns=encoder.classes_).to_csv('mlp_background_confusion_matrix.csv')

pd.DataFrame({'class':encoder.classes_,'f1':per_class_f1}).to_csv('mlp_background_per_class_f1.csv',index=False)

joblib.dump(model,'mlp_background_model.pkl')
joblib.dump(encoder,'mlp_background_encoder.pkl')
joblib.dump(scaler,'mlp_background_scaler.pkl')

print('Done')


0/21699
1000/21699
2000/21699
3000/21699
4000/21699
5000/21699
6000/21699
7000/21699
8000/21699
9000/21699
10000/21699
11000/21699
12000/21699
13000/21699
14000/21699
15000/21699
16000/21699
17000/21699
18000/21699
19000/21699
20000/21699
21000/21699
Training MLP...
Accuracy: 0.7313333333333333
Macro F1: 0.7272031233017233
Weighted F1: 0.7618633044886736
              precision    recall  f1-score   support

   agitation       0.24      0.63      0.35       390
  background       1.00      1.00      1.00       300
  clustering       0.79      0.75      0.77       870
      normal       0.89      0.71      0.79      2940

    accuracy                           0.73      4500
   macro avg       0.73      0.77      0.73      4500
weighted avg       0.82      0.73      0.76      4500

Done
